# Asian Scrollie — Segmentation Viewer

Slice-by-slice viewer for algorithms run on the **MRI_data_asian** dataset.

Ground truth is `mask_muscles.nii.gz` (per-subject/region, labels 1–13 for Thigh, variable for Calf).

**How to use:**
1. Run all cells.
2. Pick an algorithm, then a subject+region stack.
3. Toggle **Show GT** to overlay the ground-truth muscle labels.
4. Drag the slice slider.

In [1]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox
from IPython.display import display

In [2]:
# ── Label maps ────────────────────────────────────────────────────────────────

# GT label map for Asian dataset (from thigh_muscle_segmentation_labels.json)
ASIAN_GT_LABELS = {
    1:  ('rectus_femoris',     np.array([  0, 255,   0]) / 255),
    2:  ('vastus_lateralis',   np.array([255,   0,   0]) / 255),
    3:  ('vastus_intermedius', np.array([  0, 139, 139]) / 255),
    4:  ('vastus_medialis',    np.array([255, 136,   0]) / 255),
    5:  ('sartorius',          np.array([  0,   0, 255]) / 255),
    6:  ('gracilis',           np.array([255, 255,   0]) / 255),
    7:  ('biceps_femoris',     np.array([255,   0, 255]) / 255),
    8:  ('semitendinosus',     np.array([210, 180, 140]) / 255),
    9:  ('semimembranosus',    np.array([190,  83,  83]) / 255),
    10: ('adductor_brevis',    np.array([106,  90, 205]) / 255),
    11: ('adductor_longus',    np.array([  0, 255, 255]) / 255),
    12: ('adductor_magnus',    np.array([255, 124, 128]) / 255),
    13: ('gluteus_maximus',    np.array([255, 228, 225]) / 255),
}

# MuscleMap WB model — 26 labels (13 muscle groups × L/R), 7xxx scheme
MM_WB_LABELS = {
    7101: 'Vastus_Lateralis_L',         7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L',       7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',          7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',           7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',                7142: 'Sartorius_R',
    7151: 'Gracilis_L',                 7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',          7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',           7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_LongHead_L',  7182: 'Biceps_Femoris_LongHead_R',
    7191: 'Biceps_Femoris_ShortHead_L', 7192: 'Biceps_Femoris_ShortHead_R',
    7201: 'Adductor_Magnus_L',          7202: 'Adductor_Magnus_R',
    7211: 'Adductor_Longus_L',          7212: 'Adductor_Longus_R',
    7221: 'Adductor_Brevis_L',          7222: 'Adductor_Brevis_R',
}

# MuscleMap Thigh model — dedicated model, uses 1-28 label scheme (from Zenodo 19633000)
# Labels 27/28 are femur (bone), not muscle — included for display completeness.
MM_THIGH_LABELS = {
    1:  'Vastus_Lateralis_L',         2:  'Vastus_Lateralis_R',
    3:  'Vastus_Intermedius_L',       4:  'Vastus_Intermedius_R',
    5:  'Vastus_Medialis_L',          6:  'Vastus_Medialis_R',
    7:  'Rectus_Femoris_L',           8:  'Rectus_Femoris_R',
    9:  'Sartorius_L',                10: 'Sartorius_R',
    11: 'Gracilis_L',                 12: 'Gracilis_R',
    13: 'Semimembranosus_L',          14: 'Semimembranosus_R',
    15: 'Semitendinosus_L',           16: 'Semitendinosus_R',
    17: 'Biceps_Femoris_LongHead_L',  18: 'Biceps_Femoris_LongHead_R',
    19: 'Biceps_Femoris_ShortHead_L', 20: 'Biceps_Femoris_ShortHead_R',
    21: 'Adductor_Magnus_L',          22: 'Adductor_Magnus_R',
    23: 'Adductor_Longus_L',          24: 'Adductor_Longus_R',
    25: 'Adductor_Brevis_L',          26: 'Adductor_Brevis_R',
    27: 'Femur_L',                    28: 'Femur_R',
}

# MuSeg-AI thigh-model3 — 13 bilateral muscle groups, labels 1-13
MUSEG_LABELS = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

print('Label maps defined.')

Label maps defined.


In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
# To add a new algorithm: add an entry to ALGORITHMS below.
# seg_dir must contain files in {subject}/{region}/ subdirectories.
# 'glob' overrides the default '*_dseg*' filename pattern (needed for NPZ algorithms).

EVAL_DIR  = r'C:\Projects\dissector\eval_notebooks'
DATA_ROOT = os.path.join(EVAL_DIR, 'MRI_data_asian', 'MRI_data')

ALGORITHMS = {
    'MuscleMap WB (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb', 'asian_segs_water'),
        'fmt':       'nifti',
        'label_map': MM_WB_LABELS,
    },
    'MuscleMap Thigh (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_thigh', 'asian_segs_water'),
        'fmt':       'nifti',
        'label_map': MM_THIGH_LABELS,
    },
    'MedCLIP-SAMv2 Text+Boxes (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medclipsamv2textboxes', 'asian_segs_water'),
        'fmt':       'npz',
        'label_map': None,
        'glob':      '*_mcsam2textboxes.npz',
    },
    'Dafne (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'dafne', 'asian_segs_water'),
        'fmt':       'npz',
        'label_map': None,
        'glob':      '*dafne*.npz',
    },
    'MedSegDiff': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medsegdiff', 'asian_segs'),
        'fmt':       'npz',
        'label_map': None,
        'glob':      'Thigh_seg.npz',
    },
    'MuSeg (dixon)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'museg', 'asian_segs', 'dixon_based'),
        'fmt':       'nifti',
        'label_map': MUSEG_LABELS,
        'glob':      'Thigh_seg.nii.gz',
    },
}

# Only show algorithms whose seg_dir exists and has at least one matching file
def _count(cfg):
    return len(glob.glob(os.path.join(
        cfg['seg_dir'], '*', '*', cfg.get('glob', '*_dseg*')
    )))

AVAILABLE = {
    name: cfg for name, cfg in ALGORITHMS.items()
    if os.path.isdir(cfg['seg_dir']) and _count(cfg) > 0
}

print(f'Available algorithms ({len(AVAILABLE)}/{len(ALGORITHMS)})')
for name, cfg in AVAILABLE.items():
    print(f'  {name}: {_count(cfg)} stacks')

Available algorithms (6/6)
  MuscleMap WB (water): 50 stacks
  MuscleMap Thigh (water): 25 stacks
  MedCLIP-SAMv2 Text+Boxes (water): 27 stacks
  Dafne (water): 25 stacks
  MedSegDiff: 25 stacks
  MuSeg (dixon): 25 stacks


In [4]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_norm(path):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def build_overlay_from_int(seg_arr, label_map, alpha=0.5):
    """NIfTI integer labels → (D,H,W,4) RGBA using label_map colours."""
    present = {k: v for k, v in label_map.items() if np.any(seg_arr == k)}
    seq     = {i: (orig, info) for i, (orig, info) in enumerate(present.items(), 1)}
    n       = max(len(seq), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for i, (orig, info) in seq.items():
        if isinstance(info, tuple):   # ASIAN_GT_LABELS: (name, colour)
            name, colour = info
            c = (*colour, alpha)
        else:                         # integer label map: name string
            name = info
            c = (*cmap(i - 1)[:3], alpha)
        rgba[seg_arr == orig] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def build_overlay_from_npz(data, alpha=0.5):
    """NPZ string-keyed masks → (D,H,W,4) RGBA."""
    names   = list(data.files)
    n       = max(len(names), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    shape   = data[names[0]].shape
    rgba    = np.zeros((*shape, 4), dtype=np.float32)
    patches = []
    for i, name in enumerate(names, 1):
        c = (*cmap(i - 1)[:3], alpha)
        rgba[data[name] > 0] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def load_seg(seg_path, cfg):
    if cfg['fmt'] == 'nifti':
        seg_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(seg_path))).astype(np.int32)
        return build_overlay_from_int(seg_arr, cfg['label_map'])
    else:
        return build_overlay_from_npz(np.load(seg_path))


def load_gt(subject, region):
    path = os.path.join(DATA_ROOT, subject, region, 'mask_muscles.nii.gz')
    if not os.path.exists(path):
        raise FileNotFoundError(f'GT not found: {path}')
    seg_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.int32)
    return build_overlay_from_int(seg_arr, ASIAN_GT_LABELS)


def get_stacks(algo_name):
    """Return {'{subject}_{region}': seg_path} for all results found.
    Uses cfg['glob'] as the filename pattern if provided, else '*_dseg*'."""
    cfg      = AVAILABLE[algo_name]
    pattern  = cfg.get('glob', '*_dseg*')
    files    = sorted(glob.glob(os.path.join(cfg['seg_dir'], '*', '*', pattern)))
    result   = {}
    for f in files:
        parts   = f.replace('\\', '/').split('/')
        subject = parts[-3]
        region  = parts[-2]
        result[f'{subject}_{region}'] = f
    return result


print('Helpers ready.')

Helpers ready.


In [5]:
# ── Widgets ───────────────────────────────────────────────────────────────────

ALGO_OPTIONS = ['— none —'] + list(AVAILABLE)

algo1_dd   = Dropdown(options=list(AVAILABLE), description='Algorithm 1:',
                      layout=widgets.Layout(width='420px'))
algo2_dd   = Dropdown(options=ALGO_OPTIONS, value='— none —',
                      description='Algorithm 2:',
                      layout=widgets.Layout(width='420px'))
stack_dd   = Dropdown(options=[], description='Stack:',
                      layout=widgets.Layout(width='320px'))
slice_sl   = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                       layout=widgets.Layout(width='600px'))
show_gt_cb = widgets.Checkbox(value=False, description='Show GT',
                               indent=False, layout=widgets.Layout(width='120px'))
out = widgets.Output()

_cache    = {}
_gt_cache = {}


def _load(algo_name, stack_label):
    key = (algo_name, stack_label)
    if key not in _cache:
        cfg      = AVAILABLE[algo_name]
        stacks   = get_stacks(algo_name)
        seg_path = stacks[stack_label]
        subject, region = stack_label.rsplit('_', 1)
        img_norm        = load_norm(os.path.join(DATA_ROOT, subject, region, 'Water.nii.gz'))
        overlay, patches = load_seg(seg_path, cfg)
        _cache[key] = (img_norm, overlay, patches, subject, region)
    return _cache[key]


def _load_gt(algo_name, stack_label):
    key = (algo_name, stack_label)
    if key not in _gt_cache:
        _, _, _, subject, region = _load(algo_name, stack_label)
        _gt_cache[key] = load_gt(subject, region)
    return _gt_cache[key]


def render(algo1, algo2, stack_label, slice_idx, show_gt):
    if not stack_label:
        return
    try:
        img_norm, ov1, patches1, _, _ = _load(algo1, stack_label)
    except Exception as e:
        with out:
            out.clear_output(wait=True)
            print(f'Error loading {algo1}: {e}')
        return

    img = img_norm[slice_idx]

    has_algo2 = algo2 != '— none —' and algo2 in AVAILABLE
    ov2, patches2 = None, []
    if has_algo2:
        try:
            _, ov2, patches2, _, _ = _load(algo2, stack_label)
        except Exception:
            has_algo2 = False

    gt_ov, gt_patches, gt_err = None, [], None
    if show_gt:
        try:
            gt_ov, gt_patches = _load_gt(algo1, stack_label)
        except Exception as e:
            gt_err = str(e)

    panels = [('Water image', None, None)]
    if show_gt:
        panels.append(('Ground truth (mask_muscles)', gt_ov,
                        gt_patches if gt_err is None else []))
    panels.append((algo1, ov1, patches1))
    if has_algo2:
        panels.append((algo2, ov2, patches2))
    else:
        label = 'Algorithm 2 — select above' if algo2 == '— none —' \
                else f'{algo2}\n(no matching stack)'
        panels.append((label, None, None))

    n_panels  = len(panels)
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    if n_panels == 1:
        axes = [axes]

    for ax, (title, overlay, patches) in zip(axes, panels):
        ax.imshow(img, cmap='gray', origin='lower')
        if overlay is not None:
            ax.imshow(overlay[slice_idx], origin='lower')
        if patches:
            ax.legend(handles=patches, loc='lower right', fontsize=5,
                      framealpha=0.7, ncol=2)
        color = 'gray' if overlay is None and not patches else 'black'
        ax.set_title(title, fontsize=10, color=color)
        ax.axis('off')

    fig.suptitle(f'{stack_label}  —  slice {slice_idx}', fontsize=11)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()
        if gt_err:
            print(f'[GT] {gt_err}')


def _rerender(*_):
    render(algo1_dd.value, algo2_dd.value, stack_dd.value,
           slice_sl.value, show_gt_cb.value)


def on_algo1_change(change):
    _cache.clear()
    _gt_cache.clear()
    stacks = get_stacks(change['new'])
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        slice_sl.max   = _load(change['new'], stack_dd.value)[0].shape[0] - 1
        slice_sl.value = 0
    _rerender()


def on_stack_change(change):
    if change['new']:
        img_norm, *_ = _load(algo1_dd.value, change['new'])
        slice_sl.max   = img_norm.shape[0] - 1
        slice_sl.value = 0
    _rerender()


algo1_dd.observe(on_algo1_change, names='value')
algo2_dd.observe(lambda _: _rerender(), names='value')
stack_dd.observe(on_stack_change, names='value')
slice_sl.observe(lambda _: _rerender(), names='value')
show_gt_cb.observe(lambda _: _rerender(), names='value')

# Initial load
if AVAILABLE:
    stacks = get_stacks(algo1_dd.value)
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        img0, *_ = _load(algo1_dd.value, stack_dd.value)
        slice_sl.max = img0.shape[0] - 1
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, 0, show_gt_cb.value)
else:
    print('No algorithms available yet — run the Lambda notebook and download results.')

display(VBox([
    HBox([algo1_dd, algo2_dd, show_gt_cb]),
    HBox([stack_dd, slice_sl]),
    out,
]))